# 04 - Feature Engineering

**Goal:** build a clean, leakage-safe feature table for modelling.

## The key design principle

Our modelling approach (see `05_modeling.ipynb` and the README) is:

1. **Train:** features known as of the 2016 election -> predict Turnout_2021 (a genuine out-of-sample check, since we know the real 2021 answer)
2. **Forecast:** features known as of the 2021 election -> predict Turnout_2026 (the real unknown)

For this to be valid, **every feature must be computable from a single completed election's data alone** — nothing that depends on knowing the outcome we're predicting, and nothing that depends on data that won't exist yet at forecast time.

This is why every feature below is built by **one function** (`build_snapshot_features`, in `src/features.py`) that we call once on the 2016 panel and once on the 2021 panel — using the same function for both is what *guarantees* consistency between training and forecasting. If a feature can't be computed by that function from a single year alone, it doesn't belong in the model.

A concrete example of what this rules out: `Turnout_2021 - Turnout_2016` ("turnout delta") is **not** used as a feature, because it's built directly from the very outcome (`Turnout_2021`) the model is trying to predict — using it as an input would mean handing the model the answer.

**Input:** `data/processed/ward_panel.csv`, plus `SpoiltVotes` recovered directly from the raw LGE files (this was computed in `02_cleaning_and_merge.ipynb` but dropped before the panel was saved — see `src/data_loading.get_spoilt_ratios`).

**Output:**
- `data/processed/train_features.csv` — 2016 features + known Turnout_2021 target, for training and validating the model
- `data/processed/forecast_features.csv` — 2021 features, no target (Turnout_2026 hasn't happened yet), for the model to actually predict on


In [1]:
import sys
sys.path.insert(0, "..")

import pandas as pd

from src.data_loading import get_spoilt_ratios, load_ward_panel
from src.features import build_snapshot_features

pd.set_option("display.max_columns", None)

panel = load_ward_panel("../data/processed/ward_panel.csv")
panel.shape

(4344, 11)

## Step 1 - Recover SpoiltVotes from the raw files

`SpoiltRatio` (spoiled ballots / total votes cast) was computed at ward level in `02_cleaning_and_merge.ipynb` but not included in the final `ward_panel.csv`. Rather than ask for the upstream notebook to be rerun, we recompute it directly here using the same aggregation logic (now pulled out into `src/data_loading.to_ward_level` so it isn't duplicated a third time).

In [2]:
spoilt_2016 = get_spoilt_ratios("../data/raw/LGE2016")
spoilt_2021 = get_spoilt_ratios("../data/raw/LGE2021")

print(spoilt_2016.shape, spoilt_2021.shape)
spoilt_2016.head()

(4392, 3) (4468, 3)


,Province,Ward,SpoiltRatio
0,Eastern Cape,Ward 29200001,0.015832
1,Eastern Cape,Ward 29200002,0.020352
2,Eastern Cape,Ward 29200003,0.010044
3,Eastern Cape,Ward 29200004,0.004041
4,Eastern Cape,Ward 29200005,0.019725


## Step 2 - Build snapshot features for each election year

Calling the exact same function on 2016 and on 2021 — this symmetry is the leakage guardrail described above.

In [3]:
features_2016 = build_snapshot_features(panel, spoilt_2016, year=2016)
features_2021 = build_snapshot_features(panel, spoilt_2021, year=2021)

print("2016 snapshot features:", features_2016.shape)
features_2016.head()

2016 snapshot features: (4344, 10)


,Province,Ward,MunicipalityCode,RegisteredVoters_prior,Turnout_prior,IsMetro,LogRegisteredVoters_prior,MunicipalityAvgTurnout_prior,SpoiltRatio_prior,SnapshotYear
0,Eastern Cape,Ward 29200001,BUF,8851,0.570896,True,9.088286,0.555891,0.015832,2016
1,Eastern Cape,Ward 29200002,BUF,7794,0.466513,True,8.961109,0.558022,0.020352,2016
2,Eastern Cape,Ward 29200003,BUF,8118,0.416975,True,9.001839,0.559033,0.010044,2016
3,Eastern Cape,Ward 29200004,BUF,9175,0.674223,True,9.124238,0.553783,0.004041,2016
4,Eastern Cape,Ward 29200005,BUF,9228,0.560360,True,9.129998,0.556106,0.019725,2016


In [4]:
# Quick data-quality check before we go further: no missing values expected,
# since ward_panel.csv was already cleaned and every ward matched a SpoiltRatio.
assert features_2016.isna().sum().sum() == 0, "Unexpected missing values in 2016 features"
assert features_2021.isna().sum().sum() == 0, "Unexpected missing values in 2021 features"
print("No missing values in either feature set - OK to proceed.")

No missing values in either feature set - OK to proceed.


### What each feature means

| Feature | Description | Why it's safe (no leakage) |
|---|---|---|
| `Turnout_prior` | Turnout in the snapshot election | Known fact from a completed election |
| `RegisteredVoters_prior` / `LogRegisteredVoters_prior` | Ward size (log-scaled — ward size is right-skewed, see EDA) | Known fact from a completed election |
| `IsMetro` | Whether the ward's municipality is one of the 8 SA metros | Fixed structural fact, not outcome-dependent |
| `MunicipalityAvgTurnout_prior` | Average turnout across *other* wards in the same municipality (leave-one-out, so a ward's own turnout doesn't leak into its own feature) | Known fact from a completed election |
| `SpoiltRatio_prior` | Spoiled ballots / total votes cast | Known fact from a completed election |
| `Province` | Province name | Fixed |

Deliberately **excluded**: `Turnout_2021 - Turnout_2016` (turnout delta) and any feature built from the *target* election's outcome. Registration growth rate was considered too (see EDA notes) but excluded from this iteration since it would require a live, currently-unscraped 2026 voter registration snapshot to compute at forecast time — a good candidate to revisit as a bonus feature if time allows before submission.

## Step 3 - Assemble the training table (2016 features -> known 2021 outcome)

In [5]:
train_features = features_2016.merge(
    panel[["Province", "Ward", "Turnout_2021"]], on=["Province", "Ward"], how="left"
).rename(columns={"Turnout_2021": "Target_Turnout"})

print(train_features.shape)
train_features.head()

(4344, 11)


,Province,Ward,MunicipalityCode,RegisteredVoters_prior,Turnout_prior,IsMetro,LogRegisteredVoters_prior,MunicipalityAvgTurnout_prior,SpoiltRatio_prior,SnapshotYear,Target_Turnout
0,Eastern Cape,Ward 29200001,BUF,8851,0.570896,True,9.088286,0.555891,0.015832,2016,0.400459
1,Eastern Cape,Ward 29200002,BUF,7794,0.466513,True,8.961109,0.558022,0.020352,2016,0.441672
2,Eastern Cape,Ward 29200003,BUF,8118,0.416975,True,9.001839,0.559033,0.010044,2016,0.338520
3,Eastern Cape,Ward 29200004,BUF,9175,0.674223,True,9.124238,0.553783,0.004041,2016,0.491476
4,Eastern Cape,Ward 29200005,BUF,9228,0.560360,True,9.129998,0.556106,0.019725,2016,0.403802


In [6]:
# Sanity check: does the anchor feature behave as expected against the target?
train_features[["Turnout_prior", "RegisteredVoters_prior", "MunicipalityAvgTurnout_prior", "SpoiltRatio_prior", "Target_Turnout"]].corr()["Target_Turnout"]

Turnout_prior                   0.673403
RegisteredVoters_prior         -0.201550
MunicipalityAvgTurnout_prior    0.405475
SpoiltRatio_prior              -0.078171
Target_Turnout                  1.000000
Name: Target_Turnout, dtype: float64

These correlations should roughly match the EDA notebook's findings (Turnout_prior ~0.67 with the target) — if they don't, something went wrong in the merge and it's worth stopping to investigate before handing this to modelling.

## Step 4 - Assemble the forecast table (2021 features, no target yet)

This is what the trained model will actually be applied to, to produce the 2026 turnout forecast.

In [7]:
forecast_features = features_2021.copy()
print(forecast_features.shape)
forecast_features.head()

(4344, 10)


,Province,Ward,MunicipalityCode,RegisteredVoters_prior,Turnout_prior,IsMetro,LogRegisteredVoters_prior,MunicipalityAvgTurnout_prior,SpoiltRatio_prior,SnapshotYear
0,Eastern Cape,Ward 29200001,BUF,9589,0.400459,True,9.168372,0.451457,0.009635,2021
1,Eastern Cape,Ward 29200002,BUF,7655,0.441672,True,8.943114,0.450616,0.024253,2021
2,Eastern Cape,Ward 29200003,BUF,9961,0.338520,True,9.206433,0.452721,0.013642,2021
3,Eastern Cape,Ward 29200004,BUF,9327,0.491476,True,9.140669,0.449599,0.013089,2021
4,Eastern Cape,Ward 29200005,BUF,8732,0.403802,True,9.074750,0.451388,0.005672,2021


## Step 5 - Save both tables for Person C / `05_modeling.ipynb`

In [8]:
train_features.to_csv("../data/processed/train_features.csv", index=False)
forecast_features.to_csv("../data/processed/forecast_features.csv", index=False)

print(f"Saved {len(train_features)} rows to data/processed/train_features.csv")
print(f"Saved {len(forecast_features)} rows to data/processed/forecast_features.csv")

Saved 4344 rows to data/processed/train_features.csv
Saved 4344 rows to data/processed/forecast_features.csv


## Handoff notes for modelling

**Training table** (`train_features.csv`): one row per ward, features describe the 2016 election, `Target_Turnout` is the *actual* 2021 turnout. Use this to train and validate — e.g. an 80/20 or k-fold split on these rows gives a genuine estimate of how well 2016-snapshot features predict a future election, before ever touching the real forecast.

**Forecast table** (`forecast_features.csv`): same feature columns, describing the 2021 election, no target column (Turnout_2026 doesn't exist yet). Once the model is trained and validated on the training table, apply it to this table to produce the actual 2026 turnout forecast per ward.

**Categorical columns** (`Province`, `MunicipalityCode`, `IsMetro`) still need encoding (one-hot / target encoding) before most models will accept them — left as-is here so the encoding choice and fitted encoder can live in the modelling notebook and be applied identically to both tables.

**Known limitations to carry into the write-up:**
- `MunicipalityAvgTurnout_prior` is undefined for municipalities with only a single ward in the panel; these fall back to the ward's own turnout value rather than a true "other wards" average (rare — noted for completeness).
- Registration growth was considered but not included — would need a live 2026 voter registration snapshot (dataset #4 in the brief) which hasn't been collected. If time permits, revisit as a bonus feature.
- Two snapshot years (2016, 2021) is a thinner base for "learning trends over time" than the brief's ideal of a longer historical series — flagged as an explicit limitation rather than something to paper over.